In [1]:
from datetime import datetime
from pyspark.sql import functions as F

GOLD_SCHEMA = "gold"
MAINTENANCE_LOG_TABLE = "table_maintenance_log"
 
DEFAULT_RETENTION_HOURS = 168

StatementMeta(, 2eca753a-7fe9-4e92-b147-552ca84e470e, 3, Finished, Available, Finished, False)

In [2]:
GOLD_TABLES = [
    "gold.fact_orders",
    "gold.dim_customers",
    "gold.dim_products",
    "gold.dim_stores",
    "gold.dim_employees",
    "gold.dim_orders",
]

StatementMeta(, 2eca753a-7fe9-4e92-b147-552ca84e470e, 4, Finished, Available, Finished, False)

In [3]:
def optimize_table(table_name: str) -> str:
    
    try:
        print(f"[OPTIMIZE] {table_name} — started")
        spark.sql(f"OPTIMIZE {table_name}")
        print(f"[OPTIMIZE] {table_name} — completed")
        return "SUCCESS"
    except Exception as e:
        print(f"[OPTIMIZE] {table_name} — FAILED: {e}")
        return "FAILED"
 
 
def vacuum_table(table_name: str, retention_hours: int = DEFAULT_RETENTION_HOURS) -> str:
    
    try:
        print(f"[VACUUM] {table_name} — started (retention={retention_hours}h)")
        spark.sql(f"VACUUM {table_name} RETAIN {retention_hours} HOURS")
        print(f"[VACUUM] {table_name} — completed")
        return "SUCCESS"
    except Exception as e:
        print(f"[VACUUM] {table_name} — FAILED: {e}")
        return "FAILED"
 
 
def maintain_table(table_name: str, retention_hours: int = DEFAULT_RETENTION_HOURS) -> dict:

    print("=" * 80)
    print(f"Maintaining: {table_name}")
    print("=" * 80)
 
    optimize_status = optimize_table(table_name)
    vacuum_status = vacuum_table(table_name, retention_hours)
    execution_timestamp = datetime.now()
 
    print(f"[RESULT] {table_name}: optimize={optimize_status}, vacuum={vacuum_status}")
 
    return {
        "table_name": table_name,
        "optimize_status": optimize_status,
        "vacuum_status": vacuum_status,
        "execution_timestamp": execution_timestamp,
    }

StatementMeta(, 2eca753a-7fe9-4e92-b147-552ca84e470e, 5, Finished, Available, Finished, False)

In [4]:
 
maintenance_results = [
    maintain_table(table_name, DEFAULT_RETENTION_HOURS)
    for table_name in GOLD_TABLES
]
 

StatementMeta(, 2eca753a-7fe9-4e92-b147-552ca84e470e, 6, Finished, Available, Finished, False)

Maintaining: gold.fact_orders
[OPTIMIZE] gold.fact_orders — started
[OPTIMIZE] gold.fact_orders — completed
[VACUUM] gold.fact_orders — started (retention=168h)
[VACUUM] gold.fact_orders — completed
[RESULT] gold.fact_orders: optimize=SUCCESS, vacuum=SUCCESS
Maintaining: gold.dim_customers
[OPTIMIZE] gold.dim_customers — started
[OPTIMIZE] gold.dim_customers — completed
[VACUUM] gold.dim_customers — started (retention=168h)
[VACUUM] gold.dim_customers — completed
[RESULT] gold.dim_customers: optimize=SUCCESS, vacuum=SUCCESS
Maintaining: gold.dim_products
[OPTIMIZE] gold.dim_products — started
[OPTIMIZE] gold.dim_products — completed
[VACUUM] gold.dim_products — started (retention=168h)
[VACUUM] gold.dim_products — completed
[RESULT] gold.dim_products: optimize=SUCCESS, vacuum=SUCCESS
Maintaining: gold.dim_stores
[OPTIMIZE] gold.dim_stores — started
[OPTIMIZE] gold.dim_stores — completed
[VACUUM] gold.dim_stores — started (retention=168h)
[VACUUM] gold.dim_stores — completed
[RESULT] go

In [5]:

summary_df = spark.createDataFrame(maintenance_results).select(
    "table_name", "optimize_status", "vacuum_status", "execution_timestamp"
)
 
print("=" * 80)
print("GOLD LAYER MAINTENANCE SUMMARY")
print("=" * 80)
display(summary_df)
 
full_log_table = f"{GOLD_SCHEMA}.{MAINTENANCE_LOG_TABLE}"
print(f"[WRITE] {full_log_table}")
 
(
    summary_df.write
        .format("delta")
        .mode("append")          # keep a full history of every maintenance run
        .option("mergeSchema", "true")
        .saveAsTable(full_log_table)
)
 
failed_count = sum(
    1 for r in maintenance_results
    if r["optimize_status"] == "FAILED" or r["vacuum_status"] == "FAILED"
)
print(f"Maintenance complete: {len(maintenance_results)} table(s) processed, {failed_count} failure(s).")

StatementMeta(, 2eca753a-7fe9-4e92-b147-552ca84e470e, 7, Finished, Available, Finished, False)

GOLD LAYER MAINTENANCE SUMMARY


SynapseWidget(Synapse.DataFrame, 1c218893-81e1-453a-aadf-a1027f6c3471)

[WRITE] gold.table_maintenance_log
Maintenance complete: 6 table(s) processed, 0 failure(s).
